In [6]:
import pandas as pd

# Load the insurance data into a pandas DataFrame
df = pd.read_csv('../data/insurance_data.csv')


# Display the first 5 rows to inspect the data
display(df.head())

# Create a cleaned version of the DataFrame
# For simplicity, let's convert 'Gender' to numerical (0 for Male, 1 for Female)
df_cleaned = df.copy()
df_cleaned['Gender_Encoded'] = df_cleaned['Gender'].apply(lambda x: 1 if x == 'Female' else 0)

# Drop the original 'Gender' column if desired
df_cleaned = df_cleaned.drop(columns=['Gender'])

# Save the cleaned data to a new CSV file in the data directory
df_cleaned.to_csv('../data/insurance_data_cleaned.csv', index=False)

print("Cleaned data saved to data/insurance_data_cleaned.csv")
display(df_cleaned.head())

,CustomerID,Age,Gender,Province,VehicleType,AnnualIncome,RiskScore,AnnualPremium,Deductible,NCD,...,Claimed,ClaimAmount,TotalPremium,TotalClaims,CoverType,AutoMake,VehicleModel,CustomValueEstimate,ZipCode,TransactionDate
0,AC-100000,56,Male,Addis Ababa,Sedan,147270,61,2346,500,30,...,False,0.0,2346,0.0,Comprehensive,Lifan,620,32238,10002,2024-05-10
1,AC-100001,69,Female,Addis Ababa,SUV,74640,57,2334,500,0,...,True,9883.0,2334,9883.0,Comprehensive,Suzuki,Grand Vitara,52510,10001,2024-08-13
2,AC-100002,46,Male,Oromia,Sedan,70555,42,1697,250,20,...,False,0.0,1697,0.0,Third Party Fire & Theft,Lifan,620,26523,20001,2025-03-17
3,AC-100003,32,Female,Somali,Sedan,89398,63,2370,500,20,...,True,12134.0,2370,12134.0,Comprehensive,Toyota,Corolla,27036,40005,2025-03-17
4,AC-100004,60,Female,Tigray,SUV,78475,69,2582,500,0,...,False,0.0,2582,0.0,Comprehensive,Toyota,RAV4,58348,50002,2024-11-10


Cleaned data saved to data/insurance_data_cleaned.csv


,CustomerID,Age,Province,VehicleType,AnnualIncome,RiskScore,AnnualPremium,Deductible,NCD,PastClaims,...,ClaimAmount,TotalPremium,TotalClaims,CoverType,AutoMake,VehicleModel,CustomValueEstimate,ZipCode,TransactionDate,Gender_Encoded
0,AC-100000,56,Addis Ababa,Sedan,147270,61,2346,500,30,1,...,0.0,2346,0.0,Comprehensive,Lifan,620,32238,10002,2024-05-10,0
1,AC-100001,69,Addis Ababa,SUV,74640,57,2334,500,0,4,...,9883.0,2334,9883.0,Comprehensive,Suzuki,Grand Vitara,52510,10001,2024-08-13,1
2,AC-100002,46,Oromia,Sedan,70555,42,1697,250,20,1,...,0.0,1697,0.0,Third Party Fire & Theft,Lifan,620,26523,20001,2025-03-17,0
3,AC-100003,32,Somali,Sedan,89398,63,2370,500,20,0,...,12134.0,2370,12134.0,Comprehensive,Toyota,Corolla,27036,40005,2025-03-17,1
4,AC-100004,60,Tigray,SUV,78475,69,2582,500,0,4,...,0.0,2582,0.0,Comprehensive,Toyota,RAV4,58348,50002,2024-11-10,1


In [7]:
import numpy as np

# Calculate Claim Frequency (overall)
claim_frequency = df_cleaned['Claimed'].mean()
print(f"Overall Claim Frequency: {claim_frequency:.4f}")

# Calculate Claim Severity (overall - average claim amount given a claim occurred)
claim_severity = df_cleaned[df_cleaned['Claimed'] == True]['ClaimAmount'].mean()
print(f"Overall Claim Severity (average claim amount for claimed policies): {claim_severity:.2f}")

print("\n--- KPI Selection for Null Hypotheses ---")

print("1. H₀: There are no risk differences across provinces.")
print("   KPIs: Claim Frequency (proportion of claims) and Claim Severity (average claim amount).")

print("2. H₀: There are no risk differences between zip codes.")
print("   KPIs: Claim Frequency (proportion of claims) and Claim Severity (average claim amount).")

print("3. H₀: There is no significant margin (profit) difference between zip codes.")
print("   KPI: Margin (TotalPremium - TotalClaims).")

print("4. H₀: There is no significant risk difference between Women and Men.")
print("   KPIs: Claim Frequency (proportion of claims) and Claim Severity (average claim amount).")

Overall Claim Frequency: 0.1535
Overall Claim Severity (average claim amount for claimed policies): 8561.49

--- KPI Selection for Null Hypotheses ---
1. H₀: There are no risk differences across provinces.
   KPIs: Claim Frequency (proportion of claims) and Claim Severity (average claim amount).
2. H₀: There are no risk differences between zip codes.
   KPIs: Claim Frequency (proportion of claims) and Claim Severity (average claim amount).
3. H₀: There is no significant margin (profit) difference between zip codes.
   KPI: Margin (TotalPremium - TotalClaims).
4. H₀: There is no significant risk difference between Women and Men.
   KPIs: Claim Frequency (proportion of claims) and Claim Severity (average claim amount).


In [ ]:
# Import the hypothesis testing functions
import sys
sys.path.append('../src') # Add 'src' to the system path to import modules from it
from hypothesis_tests import chi_squared_test_for_claim_frequency, anova_test_for_claim_severity

# Initialize a list to store results for the summary table
hypothesis_results = []

print("\n--- Testing H₀: There are no risk differences across provinces ---")

# --- Test 1.1: Claim Frequency by Province (Chi-squared test) ---
print("\n1.1. Testing Claim Frequency by Province (Chi-squared test):")
chi2_results_province_freq = chi_squared_test_for_claim_frequency(df_cleaned, 'Province')
print(f"   Chi-squared Statistic: {chi2_results_province_freq['statistic']:.4f}")
print(f"   P-value: {chi2_results_province_freq['p_value']:.4f}")

p_value_province_freq = chi2_results_province_freq['p_value']
decision_province_freq = "Reject H₀" if p_value_province_freq < 0.05 else "Fail to Reject H₀"
print(f"   Decision: {decision_province_freq} (p < 0.05)")

hypothesis_results.append({
    'Hypothesis': 'H₀: No risk differences across provinces (Claim Frequency)',
    'Test Used': 'Chi-squared Test',
    'P-value': p_value_province_freq,
    'Decision': decision_province_freq
})

# --- Test 1.2: Claim Severity by Province (ANOVA test) ---
print("\n1.2. Testing Claim Severity by Province (ANOVA test):")
anova_results_province_severity = anova_test_for_claim_severity(df_cleaned, 'Province')

if anova_results_province_severity['statistic'] is not None:
    print(f"   F-statistic: {anova_results_province_severity['statistic']:.4f}")
    print(f"   P-value: {anova_results_province_severity['p_value']:.4f}")
    
    p_value_province_severity = anova_results_province_severity['p_value']
    decision_province_severity = "Reject H₀" if p_value_province_severity < 0.05 else "Fail to Reject H₀"
    print(f"   Decision: {decision_province_severity} (p < 0.05)")

    hypothesis_results.append({
        'Hypothesis': 'H₀: No risk differences across provinces (Claim Severity)',
        'Test Used': 'ANOVA Test',
        'P-value': p_value_province_severity,
        'Decision': decision_province_severity
    })
else:
    print(f"   Error: {anova_results_province_severity['error']}")
    hypothesis_results.append({
        'Hypothesis': 'H₀: No risk differences across provinces (Claim Severity)',
        'Test Used': 'ANOVA Test',
        'P-value': None,
        'Decision': 'Not enough data'
    })


# Display the current results table (will be refined later)
import pandas as pd
print("\n--- Current Hypothesis Test Results Summary ---")
display(pd.DataFrame(hypothesis_results))

ModuleNotFoundError: No module named 'hypothesis_tests'

In [16]:
%%writefile ../src/hypothesis_tests.py
import pandas as pd
from scipy.stats import chi2_contingency, f_oneway

def chi_squared_test_for_claim_frequency(df: pd.DataFrame, group_col: str) -> dict:
    """
    Performs a chi-squared test for independence on claim frequency across groups.

    Args:
        df (pd.DataFrame): The input DataFrame containing 'Claimed' and the grouping column.
        group_col (str): The name of the column to group by (e.g., 'Province', 'Gender').

    Returns:
        dict: A dictionary containing the chi-squared statistic, p-value, and degrees of freedom.
    """
    contingency_table = pd.crosstab(df[group_col], df['Claimed'])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    return {
        'statistic': chi2,
        'p_value': p_value,
        'dof': dof,
        'contingency_table': contingency_table
    }

def anova_test_for_claim_severity(df: pd.DataFrame, group_col: str) -> dict:
    """
    Performs an ANOVA test on claim severity across groups.

    Args:
        df (pd.DataFrame): The input DataFrame containing 'ClaimAmount' and the grouping column.
                           Only rows where 'Claimed' is True are considered.
        group_col (str): The name of the column to group by (e.g., 'Province', 'Gender').

    Returns:
        dict: A dictionary containing the F-statistic and p-value.
              Returns None if there are fewer than two groups with claims.
    """
    # Filter for policies with claims
    df_claimed = df[df['Claimed'] == True]

    # Get claim amounts for each group
    groups_data = [group['ClaimAmount'].values for name, group in df_claimed.groupby(group_col)]

    if len(groups_data) < 2:
        return {'statistic': None, 'p_value': None, 'error': 'Not enough groups with claims for ANOVA'}
    
    # Remove empty arrays which can occur if a group has no claims
    groups_data = [g for g in groups_data if len(g) > 0]

    if len(groups_data) < 2:
        return {'statistic': None, 'p_value': None, 'error': 'Not enough groups with claims for ANOVA after filtering empty'}

    f_statistic, p_value = f_oneway(*groups_data)
    return {
        'statistic': f_statistic,
        'p_value': p_value
    }

Overwriting ../src/hypothesis_tests.py
